# Lab 6: 状态与持久层 — State & Persistence

## 学习目标
- 实现 Session 持久化：保存和恢复对话历史
- 实现 Resume 会话恢复机制
- 实现简化版 Rewind：回退到之前的对话状态
- 理解「让 Agent 下班后还能无缝接上」的工程设计

## 对应 Harness 六层架构
本 Lab 对应第 ❶ 层：状态与持久层
- 工作空间、Artifact 存储、会话续接机制
- 作用：让 Agent 下班后还能无缝接上

## 对应 Claude Code 源码
- Session persistence: `~/.claude/sessions/{uuid}.json`
- `/resume` command: 恢复最近会话
- `/rewind` command: FileHistorySnapshot 回退
- Worktree sessions: Git 分支隔离
- Teleport: 跨机器续接（OAuth + Git）

## 痛点回顾

经过 Lab 1–5，我们的 Mini Harness 已经具备：
- ❶ 提示与引导层：系统提示词 + CLAUDE.md + Hooks
- ❷ 工具与执行层：统一 Tool 接口 + 执行管道
- ❸ 上下文与内存层：micro-compaction + 项目记忆
- ❹ 验证与安全层：权限中间件 + bash 分类器 + HITL
- ❺ 规划与协调层：AgentTool + 任务管理

**但有一个致命问题：退出程序，一切归零。**

对话历史、上下文窗口、任务状态——全部丢失。这就好比一个从不存档的员工，每天早上都要从头开始。

Lab 6 解决这个问题：**给 Agent 加上「持久化记忆」。**

In [1]:
# ==================================================
# Lab 1-5 recap (imports from utils/)
# ==================================================

import os, json, uuid, subprocess
from datetime import datetime
from utils.llm import create_harness_client, default_model

# LLM client
client = create_harness_client()
MODEL = default_model()
# Tools (Lab 2)
from utils.tools import Tool, ReadFileTool, WriteFileTool,EditFileTool, BashTool,clean_cache_control

# Permission checks (Lab 3)
from utils.permissions import SmartPermissionChecker, classify_bash

# Context manager (Lab 4)
from utils.context import ContextManager

# Project memory + hooks (Lab 1)
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner

# Initialize tools
TOOLS = [ReadFileTool(), WriteFileTool(), EditFileTool(), BashTool()]
tool_map = {t.name: t for t in TOOLS}
checker = SmartPermissionChecker()
ctx_manager = ContextManager(max_tokens=50000)

print('Lab 1-5 recap ready')
print(f'  tools: {[t.name for t in TOOLS]}')
print('  permission checker: SmartPermissionChecker')
print(f'  context manager: ContextManager (max={ctx_manager.max_tokens} tokens)')


Lab 1-5 代码回顾完成
  工具: ['read_file', 'write_file', 'edit_file', 'bash']
  权限检查器: SmartPermissionChecker
  上下文管理: ContextManager (max=50000 tokens)


## Phase A: Session 持久化存储

### 设计目标

Claude Code 把每次会话存储在 `~/.claude/sessions/{uuid}.json` 中。我们实现一个类似的 `SessionStore`：

| 功能 | 方法 |
|------|------|
| 创建新会话 ID | `new_session_id()` |
| 保存会话 | `save(session_id, messages, metadata)` |
| 加载会话 | `load(session_id)` |
| 列出所有会话 | `list_sessions()` |

### 关键挑战

Anthropic SDK 返回的消息包含 `ContentBlock`、`ToolUseBlock` 等对象，无法直接 `json.dumps`。
我们需要实现序列化适配器，把这些对象转为纯 JSON 字典。

In [2]:
# ==================================================
# Phase A: SessionStore — 会话持久化存储
# ==================================================
from pathlib import Path
class SessionStore:
    """
    会话持久化存储。
    将对话历史保存到本地 JSON 文件，支持保存、加载、列出会话。
    对应 Claude Code 的 ~/.claude/sessions/{uuid}.json
    """

    def __init__(self, store_dir: str = ".sessions"):
        self.store_dir = Path(store_dir)
        self.store_dir.mkdir(parents=True, exist_ok=True)

    def new_session_id(self) -> str:
        """生成新的会话 ID（UUID）"""
        return str(uuid.uuid4())

    @staticmethod
    def _serialize_message(msg: dict) -> dict:
        """
        序列化单条消息。
        Anthropic SDK 返回的 content 可能包含 ContentBlock 对象，
        需要转换为纯字典才能 JSON 序列化。
        """
        serialized = {"role": msg.get("role", "unknown")}
        content = msg.get("content", "")

        # 如果 content 是字符串，直接保留
        if isinstance(content, str):
            serialized["content"] = content
        # 如果 content 是列表（多个 ContentBlock）
        elif isinstance(content, list):
            serialized["content"] = [
                SessionStore._serialize_block(block) for block in content
            ]
        else:
            # 尝试用 __dict__ 或 str 处理未知类型
            serialized["content"] = str(content)

        return serialized

    @staticmethod
    def _serialize_block(block) -> dict:
        """
        序列化单个 ContentBlock。
        处理 TextBlock、ToolUseBlock、ToolResultBlock 等。
        """
        # 如果已经是 dict，直接返回
        if isinstance(block, dict):
            return block
        # 如果有 model_dump 方法（Pydantic v2）
        if hasattr(block, "model_dump"):
            return block.model_dump()
        # 如果有 __dict__
        if hasattr(block, "__dict__"):
            return {k: v for k, v in block.__dict__.items() if not k.startswith("_")}
        # 其他情况
        return {"type": "text", "text": str(block)}

    def save(self, session_id: str, messages: list, metadata: dict = None) -> str:
        """
        保存会话到磁盘。
        返回保存的文件路径。
        """
        filepath = self.store_dir / f"{session_id}.json"
        # 序列化所有消息
        serialized_messages = [self._serialize_message(m) for m in messages]

        session_data = {
            "session_id": session_id,
            "timestamp": datetime.now().isoformat(),
            "message_count": len(serialized_messages),
            "metadata": metadata or {},
            "messages": serialized_messages,
        }

        filepath.write_text(
            json.dumps(session_data, ensure_ascii=False, indent=2)
        )
        return str(filepath)

    def load(self, session_id: str) -> tuple:
        """
        加载会话。
        返回 (messages, metadata) 元组。
        """
        filepath = self.store_dir / f"{session_id}.json"
        if not filepath.exists():
            raise FileNotFoundError(f"会话不存在: {session_id}")

        data = json.loads(filepath.read_text())
        return data["messages"], data.get("metadata", {})

    def list_sessions(self) -> list:
        """
        列出所有已保存的会话，按时间倒序排列。
        返回 [{id, timestamp, message_count}, ...]
        """
        sessions = []
        for f in sorted(self.store_dir.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True):
            try:
                data = json.loads(f.read_text())
                sessions.append({
                    "id": data["session_id"],
                    "timestamp": data["timestamp"],
                    "message_count": data["message_count"],
                })
            except Exception:
                continue  # 跳过损坏的文件
        return sessions


# ---------- 测试 SessionStore ----------
store = SessionStore(".sessions")

# 模拟一段对话历史
test_messages = [
    {"role": "user", "content": "你好，请帮我查看当前目录"},
    {"role": "assistant", "content": [
        {"type": "text", "text": "我来查看当前目录的文件列表"},
        {"type": "tool_use", "id": "call_001", "name": "bash", "input": {"command": "ls"}},
    ]},
    {"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": "call_001", "content": "file1.txt\nfile2.py"},
    ]},
    {"role": "assistant", "content": "当前目录有 file1.txt 和 file2.py 两个文件。"},
]

# 保存
test_id = store.new_session_id()
saved_path = store.save(test_id, test_messages, metadata={"model": MODEL})
print(f"✓ 会话已保存: {saved_path}")

# 加载
loaded_msgs, loaded_meta = store.load(test_id)
print(f"✓ 加载成功: {len(loaded_msgs)} 条消息")
print(f"   元数据: {loaded_meta}")

# 列出
sessions = store.list_sessions()
print(f"✓ 会话列表: {len(sessions)} 个会话")
for s in sessions:
    print(f"   {s['id'][:8]}... | {s['timestamp']} | {s['message_count']} msgs")


✓ 会话已保存: .sessions/fa510be6-979d-4a2d-be12-e1a7af8308f3.json
✓ 加载成功: 4 条消息
   元数据: {'model': 'global.anthropic.claude-opus-4-6-v1'}
✓ 会话列表: 27 个会话
   fa510be6... | 2026-04-08T08:30:44.690268 | 4 msgs
   5e482035... | 2026-04-08T08:30:28.625898 | 4 msgs
   7989b4c1... | 2026-04-07T13:23:58.007579 | 30 msgs
   de22b6cf... | 2026-04-07T13:07:08.194473 | 4 msgs
   14523db4... | 2026-04-07T13:00:01.445216 | 12 msgs
   7cebb134... | 2026-04-07T12:59:11.728658 | 8 msgs
   279e5704... | 2026-04-07T12:58:34.302303 | 4 msgs
   e8d2c927... | 2026-04-07T09:55:26.804489 | 4 msgs
   f742d94c... | 2026-04-07T09:55:14.451723 | 4 msgs
   ac927a44... | 2026-04-07T07:27:38.350705 | 12 msgs
   6c61cf46... | 2026-04-07T07:25:09.769712 | 4 msgs
   1d755816... | 2026-04-07T07:22:16.556436 | 3 msgs
   a3825d57... | 2026-04-07T07:18:18.562426 | 4 msgs
   4e5ae46f... | 2026-04-07T07:17:38.015727 | 4 msgs
   c7a5d3fa... | 2026-04-07T07:13:18.463196 | 4 msgs
   ba90c4c1... | 2026-04-07T06:28:39.626907 | 12 msgs
 

## Phase B: 带持久化的 Agent 循环

### 设计目标

把 SessionStore 集成到 Agent 主循环中：

1. **启动时**：如果提供了 `resume_messages`，从上次对话继续
2. **运行中**：每次轮次自动检查权限、压缩上下文
3. **退出时**：`finally` 块中自动保存会话

这就是 Claude Code 的核心体验：无论怎么退出，会话都不会丢失。

In [ ]:
# ==================================================
# Phase B: 带持久化的 Agent 主循环（Streaming 模式）
# ==================================================

SYSTEM_PROMPT = '''你是一个编程助手，可以读写文件和执行 bash 命令。
请用中文回答。当需要操作文件或执行命令时，使用提供的工具。'''


import concurrent.futures
from threading import Lock

def run_agent_loop(
    system_prompt=SYSTEM_PROMPT,
    tools_list=TOOLS,
    user_messages=None,
    max_turns=50,
    hooks_runner=None,
    permission_checker=None,
    context_manager=None,
    session_store=None,
    session_id=None,
    resume_messages=None,
    max_workers=5
):
    """通用 Agent 运行器（消息列表驱动）。支持并行工具调用。"""

    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                elif blk.type == 'thinking':
                    d = {'type': 'thinking', 'thinking': blk.thinking}
                    if hasattr(blk, 'signature') and blk.signature:
                        d['signature'] = blk.signature
                    result.append(d)
                elif blk.type == 'redacted_thinking':
                    result.append({'type': 'redacted_thinking', 'data': blk.data})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    def _execute_single_tool(blk, tool):
        """
        执行单个工具调用，返回 (tool_use_id, result, is_error)。
        需要用户交互的（ask）不能并行，会在主线程单独处理。
        """
        # 参数校验
        validation_error = tool.validate(blk.input)
        if validation_error:
            return blk.id, f"参数校验失败: {validation_error}", True

        # Pre-hook
        inp = hooks_runner.run_pre_tool(blk.name, blk.input) if hooks_runner else blk.input
        if inp is None:
            return blk.id, "被 Hook 阻止", True

        # 权限检查（deny 可以在线程中直接处理）
        perm = permission_checker.check(blk.name, blk.input) if permission_checker else "allow"

        if perm == "deny":
            print(f"\n  🚫 [拒绝] {blk.name}({blk.input})")
            return blk.id, "权限拒绝：此操作已被禁止。", True

        if perm == "ask":
            # 标记需要主线程交互，返回特殊标志
            return blk.id, "__NEEDS_CONFIRMATION__", None

        # allow → 执行
        result = tool.execute(blk.input)
        if hooks_runner:
            result = hooks_runner.run_post_tool(blk.name, inp, result)
        return blk.id, result, False

    def _handle_confirmation(blk, tool):
        """主线程中处理需要用户确认的工具调用"""
        print(f"\n  ⚠️  [需要确认] {blk.name}")
        print(f"     参数: {blk.input}")
        confirm = input("     允许执行？(y/n): ")
        if confirm.strip().lower() in ("y", "yes"):
            inp = hooks_runner.run_pre_tool(blk.name, blk.input) if hooks_runner else blk.input
            result = tool.execute(blk.input)
            if hooks_runner and inp:
                result = hooks_runner.run_post_tool(blk.name, inp, result)
            print(f"     ✅ 已执行")
            return blk.id, result, False
        else:
            print(f"     ❌ 用户拒绝")
            return blk.id, "用户拒绝了此操作。", True

    def _process_tool_calls(tool_blocks):
        """
        处理所有 tool_use 块，自动判断是否可以并行。
        返回: 按原始顺序排列的 [(tool_use_id, result, is_error), ...]
        """
        if not tool_blocks:
            return []

        # 只有1个工具调用，直接串行
        if len(tool_blocks) == 1:
            blk = tool_blocks[0]
            tool = _tool_map.get(blk.name)
            if tool is None:
                return [(blk.id, f"未知工具 {blk.name}", True)]
            tid, res, err = _execute_single_tool(blk, tool)
            if res == "__NEEDS_CONFIRMATION__":
                return [_handle_confirmation(blk, tool)]
            return [(tid, res, err)]

        # 多个工具调用 → 并行执行
        # 第一步：分类 —— 已知/未知、是否需要确认
        results_map = {}  # blk.id → (result, is_error)
        parallel_tasks = []  # (blk, tool)
        confirmation_tasks = []  # (blk, tool) 需要主线程交互的

        for blk in tool_blocks:
            tool = _tool_map.get(blk.name)
            if tool is None:
                results_map[blk.id] = (f"未知工具 {blk.name}", True)
            else:
                # 预检权限，分流
                validation_error = tool.validate(blk.input)
                if validation_error:
                    results_map[blk.id] = (f"参数校验失败: {validation_error}", True)
                    continue

                perm = permission_checker.check(blk.name, blk.input) if permission_checker else "allow"
                if perm == "deny":
                    results_map[blk.id] = ("权限拒绝：此操作已被禁止。", True)
                    print(f"\n  🚫 [拒绝] {blk.name}({blk.input})")
                elif perm == "ask":
                    confirmation_tasks.append((blk, tool))
                else:
                    parallel_tasks.append((blk, tool))

        # 第二步：并行执行 allow 的任务
        if parallel_tasks:
            with concurrent.futures.ThreadPoolExecutor(max_workers=min(max_workers, len(parallel_tasks))) as executor:
                future_map = {}
                for blk, tool in parallel_tasks:
                    future = executor.submit(_run_tool_safe, blk, tool, hooks_runner)
                    future_map[future] = blk

                for future in concurrent.futures.as_completed(future_map):
                    blk = future_map[future]
                    try:
                        result, is_error = future.result()
                    except Exception as e:
                        result, is_error = f"工具执行异常: {e}", True
                    results_map[blk.id] = (result, is_error)

        # 第三步：串行处理需要确认的（用户交互不能并行）
        for blk, tool in confirmation_tasks:
            tid, res, err = _handle_confirmation(blk, tool)
            results_map[tid] = (res, err)

        # 第四步：按原始顺序返回
        ordered = []
        for blk in tool_blocks:
            res, err = results_map[blk.id]
            ordered.append((blk.id, res, err))
        return ordered

    def _run_tool_safe(blk, tool, hooks_runner):
        """线程安全的工具执行"""
        inp = hooks_runner.run_pre_tool(blk.name, blk.input) if hooks_runner else blk.input
        if inp is None:
            return "被 Hook 阻止", True
        result = tool.execute(blk.input)
        if hooks_runner:
            result = hooks_runner.run_post_tool(blk.name, inp, result)
        return result, False

    # ============ 主循环 ============

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    last_text = ""
    msg_index = 0
    messages = resume_messages[:] if resume_messages else []

    if resume_messages:
        print(f'已恢复 {len(resume_messages)} 条历史消息')
        
    if not user_messages:
        user_messages = []
    try:
        for turn in range(1, max_turns + 1):
            if msg_index >= len(user_messages):
                break
            user_input = user_messages[msg_index]
            msg_index += 1

            messages.append({"role": "user", "content": user_input})

            # 上下文压缩检查
            if context_manager and context_manager.should_compact(messages):
                messages = context_manager.compact(messages)
                print("  [Compaction] 上下文已压缩")

            while True:
                # 清理多余的 cache_control
                messages = clean_cache_control(messages)

                with client.messages.stream(
                    model=MODEL,
                    max_tokens=32000,
                    system=system_prompt,
                    tools=_tools_schema,
                    messages=messages,
                    thinking={"type": "adaptive"},
                ) as stream:
                    print(f'\nAssistant: ', end='', flush=True)
                    for text in stream.text_stream:
                        print(text, end='', flush=True)
                    print()
                    response = stream.get_final_message()
                messages.append({"role": "assistant", "content": _serialize_content(response.content)})

                # ---- 收集本轮所有 tool_use 块 ----
                tool_blocks = []
                for blk in response.content:
                    if blk.type == "text":
                        last_text = blk.text
                    elif blk.type == "tool_use":
                        tool_blocks.append(blk)

                if not tool_blocks:
                    # 无工具调用，本轮结束
                    break

                # ---- 并行/串行执行工具 ----
                n_tools = len(tool_blocks)
                if n_tools > 1:
                    print(f"\n  ⚡ 并行执行 {n_tools} 个工具调用")

                exec_results = _process_tool_calls(tool_blocks)

                # ---- 构建 tool_results ----
                tool_results = []
                for (tool_use_id, result, is_error), blk in zip(exec_results, tool_blocks):
                    print(f"  [{blk.name}] {str(blk.input)[:80]}")
                    print(f"   -> {str(result)[:300]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "cache_control": {"type": "ephemeral"},
                        "content": result,
                        **({"is_error": True} if is_error else {}),
                    })

                if tool_results:
                    messages.append({"role": "user", "content": tool_results})

                if response.stop_reason == "max_tokens":
                    # token 耗尽（可能截断了 tool_use），继续循环让模型重试
                    print("  [warn] max_tokens reached, continuing...")
                    continue
                if response.stop_reason != "tool_use":
                    break
    finally:
        # 无论正常退出还是异常，都自动保存会话
        if session_store and session_id:
            session_store.save(session_id, messages, metadata={'model': MODEL, 'turns': turn})
            print(f'\n  会话已自动保存: {session_id}')

    return last_text

print('run_agent_loop 就绪（Streaming 模式 + 权限检查 + 上下文压缩 + 会话持久化）')


run_agent_loop 就绪（Streaming 模式 + 权限检查 + 上下文压缩 + 会话持久化）


In [4]:
# 运行 Agent，会话自动保存

sess_store = SessionStore()
sess_id = sess_store.new_session_id()
print(f'新会话 ID: {sess_id}')

final_messages = run_agent_loop(
    user_messages=['请查看当前目录有哪些文件，并告诉我文件数量'],
    session_store=sess_store,
    session_id=sess_id,
    permission_checker=checker,
    context_manager=ctx_manager,
    max_turns=5,
)

print(f'\n结束！共 {len(final_messages)} 条消息已保存')

新会话 ID: 6834c830-145d-43ff-9e8a-6f488afb1c0c



Assistant: 

  ⚡ 并行执行 2 个工具调用
  [bash] {'command': 'ls -la'}
   -> total 504
drwxrwxr-x  6 ubuntu ubuntu   4096 Apr  7 13:31 .
drwxrwxr-x 10 ubuntu ubuntu   4096 Apr  7 02:19 ..
drwxrwxr-x  2 ubuntu ubuntu   4096 Apr  8 08:30 .sessions
-rw-rw-r--  1 ubuntu ubuntu    322 Apr  2 06:53 CLAUDE.md
drwxrwxr-x  3 ubuntu ubuntu   4096 Apr  7 13:18 gomoku
-rw-rw-r--  1 ubun
  [bash] {'command': 'ls | wc -l'}
   -> 11

Assistant: 当前目录下共有 **11 个文件/文件夹**（不包含隐藏文件），具体如下：

### 📁 文件夹（4 个）
| 名称 | 说明 |
|------|------|
| `gomoku/` | 五子棋相关目录 |
| `snake_game/` | 贪吃蛇游戏相关目录 |
| `utils/` | 工具类目录 |
| `.sessions/` | 隐藏的会话目录 |

### 📄 文件（7 个）
| 名称 | 大小 |
|------|------|
| `CLAUDE.md` | 322 B |
| `lab1_prompt_guidance.ipynb` | ~50 KB |
| `lab2_tool_execution.ipynb` | ~37 KB |
| `lab3_safety_permission.ipynb` | ~41 KB |
| `lab4_context_memory.ipynb` | ~42 KB |
| `lab5_planning_coordination.ipynb` | ~183 KB |
| `lab6_state_persistence.ipynb` | ~116 KB |
| `requirements.txt` | 44 B |

可以看出这是一个包含 6 个实验（lab1~lab6）的 Ju

## Phase C: Resume — 会话恢复

### 设计目标

Claude Code 的 `/resume` 命令可以恢复最近的会话。流程：

```
1. 列出所有已保存的会话
2. 选择最近的一个（或用户指定的）
3. 加载历史消息
4. 用 resume_messages 启动新的 Agent Loop
```

这样 Agent 就能「下班后还能无缝接上」。

In [5]:
# ==================================================
# Phase C: Resume — 会话恢复
# ==================================================

# 步骤 1: 列出已保存的会话
sessions = sess_store.list_sessions()
print("已保存的会话:")
for i, s in enumerate(sessions):
    print(f"  [{i}] {s['id'][:12]}... | {s['timestamp']} | {s['message_count']} msgs")

# 步骤 2: 恢复最近的会话
if sessions:
    target = sessions[0]  # 最近的会话
    print(f" 正在恢复会话: {target['id'][:12]}...")

    resumed_msgs, meta = sess_store.load(target["id"])
    print(f"   加载了 {len(resumed_msgs)} 条历史消息")
    print(f"   元数据: {meta}")

    # 在恢复的会话上继续对话
    resumed_messages = run_agent_loop(
        user_messages=["接上上次的对话，请帮我创建一个叫 hello.txt 的文件，内容是 Hello from resumed session"],
        session_store=sess_store,
        session_id=target["id"],  # 复用同一个 session ID
        resume_messages=resumed_msgs,
        max_turns=5,
    )

    print(f"\n✓ Resume 完成！会话现在共 {len(resumed_messages)} 条消息")
else:
    print("✗ 没有已保存的会话可恢复")


已保存的会话:
  [0] 6834c830-145... | 2026-04-08T08:30:55.320649 | 4 msgs
  [1] fa510be6-979... | 2026-04-08T08:30:44.690268 | 4 msgs
  [2] 5e482035-3ee... | 2026-04-08T08:30:28.625898 | 4 msgs
  [3] 7989b4c1-0a8... | 2026-04-07T13:23:58.007579 | 30 msgs
  [4] de22b6cf-908... | 2026-04-07T13:07:08.194473 | 4 msgs
  [5] 14523db4-0f3... | 2026-04-07T13:00:01.445216 | 12 msgs
  [6] 7cebb134-5ea... | 2026-04-07T12:59:11.728658 | 8 msgs
  [7] 279e5704-dbb... | 2026-04-07T12:58:34.302303 | 4 msgs
  [8] e8d2c927-ab6... | 2026-04-07T09:55:26.804489 | 4 msgs
  [9] f742d94c-7c8... | 2026-04-07T09:55:14.451723 | 4 msgs
  [10] ac927a44-94a... | 2026-04-07T07:27:38.350705 | 12 msgs
  [11] 6c61cf46-ce2... | 2026-04-07T07:25:09.769712 | 4 msgs
  [12] 1d755816-ade... | 2026-04-07T07:22:16.556436 | 3 msgs
  [13] a3825d57-131... | 2026-04-07T07:18:18.562426 | 4 msgs
  [14] 4e5ae46f-e5f... | 2026-04-07T07:17:38.015727 | 4 msgs
  [15] c7a5d3fa-9f6... | 2026-04-07T07:13:18.463196 | 4 msgs
  [16] ba90c4c1-8d5... 


Assistant: 
  [bash] {'command': 'pwd'}
   -> /home/ubuntu/workspace/agent_harness/labs

Assistant: 
  [write_file] {'file_path': '/home/ubuntu/workspace/agent_harness/labs/hello.txt', 'content': 
   -> 已写入 26 字符到 /home/ubuntu/workspace/agent_harness/labs/hello.txt

Assistant: 文件已成功创建！✅

- **文件名**：`hello.txt`
- **路径**：`/home/ubuntu/workspace/agent_harness/labs/hello.txt`
- **内容**：`Hello from resumed session`

现在当前目录下就有 **12 个文件/文件夹** 了，比上次多了这个新创建的 `hello.txt`。

  会话已自动保存: 6834c830-145d-43ff-9e8a-6f488afb1c0c

✓ Resume 完成！会话现在共 186 条消息


## Phase D: Rewind — 对话状态回退

### 设计目标

Claude Code 的 `/rewind` 命令可以回退到之前的对话状态，配合 `FileHistorySnapshot` 还能回滚文件修改。

我们实现一个简化版：

1. 每次 Agent 轮次结束后，保存一个 checkpoint（对话快照）
2. 用户可以 rewind 到任意轮次
3. 从该轮次继续对话

```
轮次 1 → checkpoint[1]
轮次 2 → checkpoint[2]
轮次 3 → checkpoint[3]  ← 当前

rewind_to(1) → 回到轮次 1 的状态，重新开始
```

In [6]:
# ==================================================
# Phase D: Rewind — 对话状态回退
# ==================================================

import copy


class SessionRewind:
    """
    简化版 Rewind：在每个轮次保存快照，允许回退到之前的状态。
    对应 Claude Code 的 /rewind 命令 + FileHistorySnapshot。
    """

    def __init__(self):
        self.checkpoints = []  # [(turn_number, messages_snapshot), ...]

    def checkpoint(self, turn: int, messages: list):
        """保存当前轮次的对话快照"""
        # 用 copy.deepcopy 确保快照不受后续修改影响
        snapshot = copy.deepcopy(messages)
        self.checkpoints.append((turn, snapshot))

    def rewind_to(self, turn: int) -> list:
        """回退到指定轮次，返回该轮次的消息快照"""
        for t, msgs in reversed(self.checkpoints):
            if t <= turn:
                return copy.deepcopy(msgs)
        return None

    def list_checkpoints(self) -> list:
        """列出所有可用的检查点"""
        return [t for t, _ in self.checkpoints]


# ---------- 演示 Rewind 功能 ----------

rewind = SessionRewind()

# 模拟 3 个轮次的对话
demo_msgs = []

# 轮次 1
demo_msgs.append({"role": "user", "content": "第一个问题：今天天气如何？"})
demo_msgs.append({"role": "assistant", "content": "今天天气晴朗。"})
rewind.checkpoint(1, demo_msgs)

# 轮次 2
demo_msgs.append({"role": "user", "content": "第二个问题：帮我写个文件"})
demo_msgs.append({"role": "assistant", "content": "已创建 test.txt"})
rewind.checkpoint(2, demo_msgs)

# 轮次 3
demo_msgs.append({"role": "user", "content": "第三个问题：请删除所有文件"})
demo_msgs.append({"role": "assistant", "content": "已删除所有文件（这个操作有问题！）"})
rewind.checkpoint(3, demo_msgs)

print("✓ 3 个轮次的快照已保存")
print(f"   可用检查点: {rewind.list_checkpoints()}")
print(f"   当前消息数: {len(demo_msgs)}")

# 回退到轮次 1（轮次 3 的删除操作太危险了！）
print("🔄 执行 rewind_to(1)...")
rewound_msgs = rewind.rewind_to(1)
if rewound_msgs:
    print(f"   回退成功！消息数从 {len(demo_msgs)} → {len(rewound_msgs)}")
    print(f"   最后一条消息: {rewound_msgs[-1]}")
    print("\n   现在可以从轮次 1 继续对话，避免危险操作。")
else:
    print("   回退失败")


✓ 3 个轮次的快照已保存
   可用检查点: [1, 2, 3]
   当前消息数: 6
🔄 执行 rewind_to(1)...
   回退成功！消息数从 6 → 2
   最后一条消息: {'role': 'assistant', 'content': '今天天气晴朗。'}

   现在可以从轮次 1 继续对话，避免危险操作。


---
## 完整集成示例：六层 Harness 完整运行

这是整个 Workshop 的最终集成，全部六层 Harness 同时工作：
- ❶ 提示与引导层（CLAUDE.md 注入 + Hooks）
- ❷ 工具与执行层（真实执行）
- ❸ 上下文与内存层（Compaction）
- ❹ 验证与安全层（权限检查）
- ❺ 规划与协调层（AgentTool + TaskManager — 如需要可启用）
- ❻ 状态与持久层（Session 保存 + Resume）


In [12]:
# AgentTool: 子 Agent 生成器（对应 Claude Code 的 tools/AgentTool/）
import os
from utils.project_memory import ProjectMemory
class AgentTool(Tool):
    """生成子 agent 来处理子任务。子 agent 拥有独立的 context。"""

    def __init__(self, base_tools):
        self._base_tools = base_tools

    @property
    def name(self): return "dispatch_agent"

    @property
    def description(self):
        return "派遣一个子 agent 来独立完成一个具体的子任务。子 agent 拥有独立的上下文。请提供子Agent清晰具体的任务描述。"

    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "prompt": {"type": "string", "description": "要派发给子 agent 的具体任务描述"},
            },
            "required": ["prompt"],
        }

    def execute(self, tool_input):
        task = tool_input["prompt"]
        print(f"\n  子Agent启动: {task[:80]}...")

        sub_prompt = f"{task}"
        # ❶ 项目记忆注入
        pm = ProjectMemory()
        full_prompt = pm.build_system_prompt(sub_prompt, os.getcwd())
        # 子 agent 用消息列表驱动（单条任务消息）
        result = run_agent_loop(
            system_prompt=full_prompt,
            tools_list=self._base_tools,
            user_messages=[task],
            max_turns=20,
        )

        print(f"  子Agent完成: {result[:100]}...")
        return result

print("AgentTool 就绪!")

class TaskManager:
    """简单的任务管理系统。"""
    
    def __init__(self):
        self._tasks: dict[str, dict] = {}
        self._counter = 0
    
    def create(self, subject: str, description: str = "") -> str:
        self._counter += 1
        task_id = str(self._counter)
        self._tasks[task_id] = {
            "subject": subject,
            "description": description,
            "status": "pending",
        }
        return task_id
    
    def update(self, task_id: str, status: str):
        if task_id in self._tasks:
            self._tasks[task_id]["status"] = status
    
    def list_all(self) -> list[dict]:
        return [{"id": k, **v} for k, v in self._tasks.items()]

# 全局任务管理器实例
task_mgr = TaskManager()

# ❺ SendMessage + Mailbox（来自 Lab 5）
class Mailbox:
    def __init__(self):
        self._messages = []
    def send(self, sender, receiver, content):
        self._messages.append({'sender': sender, 'receiver': receiver, 'content': content, 'time': datetime.now().isoformat()})
    def receive(self, receiver):
        msgs = [m for m in self._messages if m['receiver'] == receiver]
        self._messages = [m for m in self._messages if m['receiver'] != receiver]
        return msgs
    def all_messages(self):
        return list(self._messages)

mailbox = Mailbox()
# 任务创建工具
class TaskCreateTool(Tool):
    @property
    def name(self): return "create_task"
    @property
    def description(self): return "创建一个子任务来跟踪工作进度"
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "subject": {"type": "string", "description": "任务标题"},
                "description": {"type": "string", "description": "任务详情"},
            },
            "required": ["subject"],
        }
    def execute(self, tool_input):
        tid = task_mgr.create(tool_input["subject"], tool_input.get("description", ""))
        return f"任务 #{tid} 已创建: {tool_input['subject']}"

# 任务列表工具
class TaskListTool(Tool):
    @property
    def name(self): return "list_tasks"
    @property
    def description(self): return "列出所有任务及其状态"
    @property
    def input_schema(self):
        return {"type": "object", "properties": {}}
    def execute(self, tool_input):
        tasks = task_mgr.list_all()
        if not tasks:
            return "当前没有任务。"
        lines = []
        for t in tasks:
            icon = "✅" if t["status"] == "completed" else "⏳" if t["status"] == "in_progress" else "📋"
            lines.append(f"{icon} #{t['id']} [{t['status']}] {t['subject']}")
        return "\n".join(lines)

# 任务更新工具
class TaskUpdateTool(Tool):
    @property
    def name(self): return "update_task"
    @property
    def description(self): return "更新任务状态（pending → in_progress → completed）"
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "任务 ID"},
                "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]},
            },
            "required": ["task_id", "status"],
        }
    def execute(self, tool_input):
        task_mgr.update(tool_input["task_id"], tool_input["status"])
        return f"任务 #{tool_input['task_id']} 状态已更新为 {tool_input['status']}"

class SendMessageTool(Tool):
    @property
    def name(self): return 'send_message'
    @property
    def description(self): return '发送消息给其他 agent，用于 Team Lead 与子 agent 之间的通信。'
    @property
    def input_schema(self):
        return {
            'type': 'object',
            'properties': {
                'to': {'type': 'string', 'description': '接收者名称'},
                'message': {'type': 'string', 'description': '消息内容'},
            },
            'required': ['to', 'message'],
        }
    def execute(self, tool_input):
        mailbox.send('agent', tool_input['to'], tool_input['message'])
        return f'消息已发送给 {tool_input["to"]}'


AgentTool 就绪!


In [13]:
# === 最终集成：全部六层 Harness ===
# 从 utils/ 导入所有前面 Lab 的实现

import os
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner
from utils.context import ContextManager
from utils.permissions import SmartPermissionChecker

# ❶ 项目记忆
pm = ProjectMemory()
base_prompt = '你是一个 Team Lead Agent。用中文回答。可以使用工具完成任务，也可以派遣子 agent。'
full_prompt = pm.build_system_prompt(base_prompt, os.getcwd())

# ❶ Hooks: 全流程审计
audit = []
def audit_hook(name, inp):
    audit.append({'tool': name, 'time': datetime.now().isoformat()})
    print(f'  [Hook] {name}')
    return inp

hooks = HookRunner()
hooks.register_pre_tool(audit_hook)

# ❸ 权限检查
perm_checker = SmartPermissionChecker()

# ❹ 上下文管理
ctx = ContextManager(max_tokens=50000)


# 基础工具（子 agent 可用）
base_tools = [ReadFileTool(), WriteFileTool(), BashTool(),EditFileTool()]

# 主 agent 的完整工具集
all_tools = base_tools + [
    AgentTool(base_tools=base_tools),  # 子 agent 只能用基础工具
    TaskCreateTool(),
    TaskListTool(),
    TaskUpdateTool(),
    SendMessageTool(),
]

# ❻ Session 持久化
store = SessionStore()
sid = store.new_session_id()

print(f'会话 ID: {sid}')
print(f'System prompt: {len(full_prompt)} 字符')
print(f'工具: {[t.name for t in all_tools]}')

print('\n' + '=' * 60)
print('六层 Harness 完整运行')
print('  ❶ 提示与引导（CLAUDE.md + Hooks 审计）')
print('  ❷ 工具与执行（read_file / write_file / edit_file/ bash）')
print('  ❸ 验证与安全（权限检查 + Bash 分类器）')
print('  ❹ 上下文与内存（Compaction）')
print('  ❺ 规划与协调（SendMessage）')
print('  ❻ 状态与持久（Session 自动保存）')
print('=' * 60)

run_agent_loop(
    system_prompt=full_prompt,
    tools_list=all_tools,
    user_messages=[
    """创建一个agent teams，包含produc manager agent，coding agent，code reviwer agent，test agent,帮我开发一个贪吃蛇html小游戏.保存到snake_game/中
    ## 开发流程：
    1. produc manager agent生成设计需求文档
    2. coding agent根据需求文档开发代码，完成之后，发送给test agent run lint test 和 code reviwer审查，, 如有问题直接反馈修改
    3. code reviwer反馈意见给coding agent进行修改，检查无误之后通过
    4. test agent run lint test, 如有问题直接告诉coding agent修改，检查无误之后通过
       """
    ],
    hooks_runner=hooks,
    permission_checker=perm_checker,
    context_manager=ctx,
    session_store=store,
    session_id=sid,
)

# 验证
print(f'\n--- 持久化验证 ---')
for s in store.list_sessions():
    # print(s)
    print(f'  {s["id"]}  |  {s["timestamp"]}  |  {s["message_count"]} 条消息')

print(f'\n--- 审计记录 ({len(audit)} 条) ---')
for a in audit:
    print(f'  {a["tool"]} @ {a["time"]}')



扫描目录: /home/ubuntu/workspace/agent_harness/labs
  环境信息已注入
  发现记忆文件: /home/ubuntu/workspace/agent_harness/labs/CLAUDE.md
  项目记忆已注入 system prompt
会话 ID: 936022a2-91c4-4712-95a2-1b23ebad1bca
System prompt: 815 字符
工具: ['read_file', 'write_file', 'bash', 'edit_file', 'dispatch_agent', 'create_task', 'list_tasks', 'update_task', 'send_message']

六层 Harness 完整运行
  ❶ 提示与引导（CLAUDE.md + Hooks 审计）
  ❷ 工具与执行（read_file / write_file / edit_file/ bash）
  ❸ 验证与安全（权限检查 + Bash 分类器）
  ❹ 上下文与内存（Compaction）
  ❺ 规划与协调（SendMessage）
  ❻ 状态与持久（Session 自动保存）



Assistant: 好的！我来组建一个 Agent 团队，按照流程开发贪吃蛇 HTML 小游戏。让我先创建任务追踪，然后依次派遣各个 Agent。

## 🚀 启动 Agent 团队开发流程
  [Hook] bash

  ⚠️  [需要确认] bash
     参数: {'command': 'mkdir -p /home/ubuntu/workspace/agent_harness/labs/snake_game'}
  [Hook] bash
     ✅ 已执行
  [bash] {'command': 'mkdir -p /home/ubuntu/workspace/agent_harness/labs/snake_game'}
   -> (无输出)

Assistant: 

  ⚡ 并行执行 4 个工具调用
  [Hook] create_task
  [Hook] create_task
  [Hook] create_task
  [Hook] create_task
  [create_task] {'subject': '📋 Step 1: Product Manager - 生成贪吃蛇游戏设计需求文档', 'description': 'Product
   -> 任务 #1 已创建: 📋 Step 1: Product Manager - 生成贪吃蛇游戏设计需求文档
  [create_task] {'subject': '💻 Step 2: Coding Agent - 根据需求文档开发贪吃蛇游戏代码', 'description': 'Coding A
   -> 任务 #2 已创建: 💻 Step 2: Coding Agent - 根据需求文档开发贪吃蛇游戏代码
  [create_task] {'subject': '🔍 Step 3: Code Reviewer - 代码审查', 'description': 'Code Reviewer Agen
   -> 任务 #3 已创建: 🔍 Step 3: Code Reviewer - 代码审查
  [create_task] {'subject': '🧪 Step 4: Test Agent - Lint 测试', 'description': 'Test Agent 

## 源码对照：Mini Harness vs Claude Code

### Session 持久化

| 特性 | Mini Harness (Lab 6) | Claude Code |
|------|---------------------|-------------|
| 存储位置 | `.sessions/{uuid}.json` | `~/.claude/sessions/{uuid}.json` |
| 序列化 | 手动处理 ContentBlock | `MessageHistory.serialize()` |
| 保存时机 | `finally` 块自动保存 | 每次 API 调用后自动保存 |
| 元数据 | model, turns | model, cwd, git branch, timestamp |

### Resume 机制

| 特性 | Mini Harness | Claude Code |
|------|-------------|-------------|
| 触发方式 | `resume_messages` 参数 | `/resume` 命令 |
| 会话选择 | `list_sessions()[0]` | 交互式列表选择 |
| 续接能力 | 对话历史 | 对话 + 文件状态 + Git 状态 |

### Rewind 机制

| 特性 | Mini Harness | Claude Code |
|------|-------------|-------------|
| 实现方式 | 内存中的消息快照 | `FileHistorySnapshot` 文件回滚 |
| 回退粒度 | 按轮次 | 按轮次 + 文件变更 |
| 文件回滚 | 不支持 | 支持（恢复被修改的文件） |

### Claude Code 更高级的持久化特性

- **Worktree Sessions**: 每个 Git worktree 有独立的会话，分支隔离不干扰
- **Teleport**: 跨机器续接，通过 OAuth + Git 同步会话状态
- **Artifact 存储**: 工具产生的文件差异、截图等都会存储
- **增量保存**: 只保存变化的部分，而不是每次全量写入

## Workshop 全局总结

### 六层架构全景

```
Lab 1: ❶ 提示与引导层 → system prompt + CLAUDE.md + Hooks
Lab 2: ❷ 工具与执行层 → 统一 Tool 接口 + 执行管道
Lab 3: ❸ 上下文与内存层 → micro-compaction + 项目记忆
Lab 4: ❹ 验证与安全层 → 权限中间件 + bash 分类器 + HITL
Lab 5: ❺ 规划与协调层 → AgentTool + 任务管理
Lab 6: ❻ 状态与持久层 → Session 持久化 + Resume + Rewind
```

### 核心结论

> **模型决定能力上限，Harness 决定工程下限和交付能力。**

LLM 本身只是一个「能力引擎」，Harness 才是把这个引擎变成可靠产品的关键。

### 完整架构映射表

| 层次 | Mini Harness 实现 | Claude Code 源码 |
|------|------------------|----------------|
| ❶ 提示与引导 | system prompt 字符串 + CLAUDE.md 读取 | `buildSystemPrompt()` + `claudeMdTool` + Hooks |
| ❷ 工具与执行 | Tool 抽象基类 + `to_anthropic_tool()` | `ToolDefinition` + `ToolExecutor` pipeline |
| ❸ 上下文与内存 | `ContextManager.compact_if_needed()` | `MicroCompaction` + `ProjectMemory` |
| ❹ 验证与安全 | `SmartPermissionChecker` + `classify_bash()` | `PermissionMiddleware` + `BashClassifier` + HITL |
| ❺ 规划与协调 | `AgentTool` + `TaskManager` | `AgentTool` + Orchestrator + Task Planner |
| ❻ 状态与持久 | `SessionStore` + `SessionRewind` | `SessionPersistence` + `/resume` + `/rewind` + Worktree + Teleport |

### 从 Mini Harness 到生产级 Harness

| 维度 | Mini Harness | 生产级 (Claude Code) |
|------|-------------|-------------------|
| 代码量 | ~300 行 | ~50,000 行 |
| 工具数 | 3 个 | 20+ 个 |
| 并发 | 单线程 | 多进程 + 异步 |
| 安全 | 基础规则 | 多层防御 + 审计日志 |
| 持久化 | JSON 文件 | JSON + Git + 跨机器同步 |
| 多 Agent | 基础委托 | 分层编排 + 任务规划 |

### 下一步

恭喜完成全部 6 个 Lab！你现在已经理解了 Agent Harness 的核心架构。
接下来可以：

1. **阅读 Claude Code 源码**: 带着这 6 层架构的视角去看实际代码
2. **扩展 Mini Harness**: 加入更多工具、更智能的压缩、更完善的安全策略
3. **构建自己的 Harness**: 针对特定场景设计完整的 Agent 系统